In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import numpy as np

In [14]:
from bayesgpt.simulators import ModelVariant, Tokenizer
from bayesgpt.simulators.benchmarks import SuperDDM, StandardDDM, CollapsingBoundDDM

### Metas

In [15]:
num_samples = 1000  # Global number of samples per model variant

In [16]:
# Define modulation function for context-dependent parameters in SuperDDM
def modulation(params, context):
    """Adjust drift rate based on context (e.g., stimulus strength)."""
    params = params.copy()
    if "v" in params:
        params["v"] = params["v"] * (1 + context[0])  # Scale drift rate
    elif "v_components" in params:
        params["v_components"] = params["v_components"] * (1 + context[0])
    elif "v_schedule" in params:
        params["v_schedule"] = params["v_schedule"] * (1 + context[0])
    return params

In [17]:
# Common tokenizer parameters
parameter_names = [
    "v",
    "a",
    "z",
    "tau",
    "sigma",
    "angle",
    "s_v",
    "s_z",
    "s_tau",
    "v_components",
    "p_components",
    "v_schedule",
    "t_schedule"
]

### DDM Variants

In [18]:
super_ddm_params = parameter_names

In [19]:
fixed_parameters = {
    "p_components": np.array([0.6, 0.4]),  # Mixture probabilities
    "t_schedule": np.array([0.0, 0.5])  # Time points for scheduled drifts
}
free_parameters = {
    "a": lambda c: np.random.uniform(0.8, 1.2, 1),  # Decision boundary
    "sigma": lambda c: np.random.uniform(0.05, 0.15, 1),  # Diffusion noise
    "s_v": lambda c: np.random.uniform(0.01, 0.1, 1),  # Drift rate noise
    "angle": lambda c: np.random.uniform(0.0, 0.05, 1),  # Boundary collapse
    "s_z": lambda c: np.random.uniform(0.005, 0.02, 1),  # Starting point noise
    "s_tau": lambda c: np.random.uniform(0.005, 0.02, 1),  # Non-decision time noise
    "v_components": lambda c: np.random.randn(2) * 0.5,  # Mixture drift rates
    "v_schedule": lambda c: np.random.randn(2) * 0.5,  # Scheduled drift rates
    "z": lambda c: np.random.uniform(0.4, 0.6, 1),  # Starting point
    "tau": lambda c: np.random.uniform(0.1, 0.3, 1)  # Non-decision time
}
parameter_dims = {
    "a": 1, "v": 1, "sigma": 1, "s_v": 1, "z": 1, "tau": 1, "angle": 1,
    "s_z": 1, "s_tau": 1, "v_components": 2, "p_components": 2,
    "v_schedule": 2, "t_schedule": 2
}

In [20]:
tokenizer_super_mixture = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_components", "p_components", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_super_schedule = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_schedule", "t_schedule", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_standard = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_collapsing = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)

In [21]:
model_variant_mixture = ModelVariant(
    name="super_ddm_mixture",
    model=SuperDDM,
    tokenizer=tokenizer_super_mixture,
    num_samples=num_samples
)
model_variant_schedule = ModelVariant(
    name="super_ddm_schedule",
    model=SuperDDM,
    tokenizer=tokenizer_super_schedule,
    num_samples=num_samples
)
model_variant_standard = ModelVariant(
    name="standard_ddm",
    model=StandardDDM,
    tokenizer=tokenizer_standard,
    num_samples=num_samples
)
model_variant_collapsing = ModelVariant(
    name="collapsing_bound_ddm",
    model=CollapsingBoundDDM,
    tokenizer=tokenizer_collapsing,
    num_samples=num_samples
)

In [22]:
# Cell 3: Run simulations and print results
context = np.array([0.5], dtype=np.float32)  # Single simulation context
result_super_mixture = model_variant_mixture.sample(context=context)
result_super_schedule = model_variant_schedule.sample(context=context)
result_standard = model_variant_standard.sample(context=context)
result_collapsing = model_variant_collapsing.sample(context=context)

In [23]:
def print_variant_results(variant: ModelVariant, result: dict, tokenizer: Tokenizer, num_samples: int):
    """Print simulation results and summary statistics for a ModelVariant."""
    print(f"\n{variant.name} Results:")
    print("Variant Name:", result["variant_name"])
    print("Simulated Data Keys:", result["sim_data"].keys())
    print("Reaction Times (first 5):", result["sim_data"]["rts"][:5])
    print("Choices (first 5):", result["sim_data"]["choices"][:5])
    print("Full Parameters (shape):", result["full_params"].shape)
    print("Inference Conditions (shape):", result["inference_conditions"].shape)

    # Summarize results using the model's summarize method
    summary = variant.model.summarize(
        outputs=result["sim_data"],
        quantile_levels=[0.1, 0.3, 0.5, 0.7, 0.9],
        by_choice=True,
        tau=np.full(num_samples, result["full_params"][
            tokenizer.parameter_slices["tau"]][0], dtype=np.float32)
    )
    print(f"{variant.name} Summary:")
    print("Invalid Rate:", summary["invalid_rate"])
    print("RT Quantiles:", summary["rt_quantiles"])
    print("RT Quantiles by Choice:\n", summary["rt_quantiles_by_choice"])
    print("Decision Time Quantiles:", summary["dt_quantiles"])
    print("Decision Time Quantiles by Choice:\n", summary["dt_quantiles_by_choice"])

In [24]:
print_variant_results(model_variant_mixture, result_super_mixture, tokenizer_super_mixture, num_samples)
print_variant_results(model_variant_schedule, result_super_schedule, tokenizer_super_schedule, num_samples)
print_variant_results(model_variant_standard, result_standard, tokenizer_standard, num_samples)
print_variant_results(model_variant_collapsing, result_collapsing, tokenizer_collapsing, num_samples)


super_ddm_mixture Results:
Variant Name: super_ddm_mixture
Simulated Data Keys: dict_keys(['rts', 'choices', 'context'])
Reaction Times (first 5): [5.132021  1.362164  5.1001644 3.4585688 5.3733125]
Choices (first 5): [0. 1. 0. 0. 0.]
Full Parameters (shape): (17,)
Inference Conditions (shape): (34,)
super_ddm_mixture Summary:
Invalid Rate: 0.0
RT Quantiles: [1.4992738 1.9545026 3.6501951 4.2287455 4.8807898]
RT Quantiles by Choice:
 [[3.57672   3.9790776 4.25916   4.5668354 5.110844 ]
 [1.3436828 1.5604334 1.7680819 1.9485234 2.4087503]]
Decision Time Quantiles: [1.222015  1.6772438 3.3729362 3.9514866 4.603531 ]
Decision Time Quantiles by Choice:
 [[3.2994611 3.7018187 3.9819012 4.2895765 4.8335853]
 [1.066424  1.2831746 1.4908231 1.6712646 2.1314914]]

super_ddm_schedule Results:
Variant Name: super_ddm_schedule
Simulated Data Keys: dict_keys(['rts', 'choices', 'context'])
Reaction Times (first 5): [2.5787792 2.1495948 1.885435  1.6242867 1.4163078]
Choices (first 5): [1. 1. 1. 1. 